# Correlação Intra Classe para verificar consistência entre as medidas.

### Bibliotecas utilizadas

In [1]:
import pandas as pd
import pingouin as pg

### Lendo a base

In [2]:
caminho = r'C:\Users\saulosgil\OneDrive\Documentos\functional_tests_blinding\data\data_blinding.xlsx'
df = pd.read_excel(caminho)

df.head()

,id,idade,estado_civil,sexo,raca,renda,baixa_visao,cego,comorbidades,medicamentos,peso_kg,estatura_cm,imc,tug_1,tug_2,ts_1,ts_2
0,5,19,Solteiro,Feminino,Branco,Até 1 salário mínimo,1,0,Pre diabete,Loratadina e clenil,55.0,158,22.031726,8.65,6.54,15,18
1,9,18,Solteiro,Feminino,Preto,Até 1 salário mínimo,1,0,Ansiedade,Nenhum,70.0,171,23.938990,6.09,5.51,11,13
2,10,62,Divorciado,Masculino,Branco,1 – 2 salários mínimos,1,0,Nenhuma comorbidade,Nenhum nenhuma,85.0,180,26.234568,7.89,9.77,14,15
3,11,42,Solteiro,Masculino,Preto,1 – 2 salários mínimos,1,0,Nenhuma comorbidade,Não,105.0,168,37.202381,12.80,8.75,12,15
4,17,42,Solteiro,Masculino,Preto,Até 1 salário mínimo,1,0,Nenhuma comorbidade,Não,75.0,165,27.548209,10.00,8.10,10,12


## **Selecionar apenas os cegos**

In [3]:
df = df[df['baixa_visao'] == 1]
df.head()

,id,idade,estado_civil,sexo,raca,renda,baixa_visao,cego,comorbidades,medicamentos,peso_kg,estatura_cm,imc,tug_1,tug_2,ts_1,ts_2
0,5,19,Solteiro,Feminino,Branco,Até 1 salário mínimo,1,0,Pre diabete,Loratadina e clenil,55.0,158,22.031726,8.65,6.54,15,18
1,9,18,Solteiro,Feminino,Preto,Até 1 salário mínimo,1,0,Ansiedade,Nenhum,70.0,171,23.938990,6.09,5.51,11,13
2,10,62,Divorciado,Masculino,Branco,1 – 2 salários mínimos,1,0,Nenhuma comorbidade,Nenhum nenhuma,85.0,180,26.234568,7.89,9.77,14,15
3,11,42,Solteiro,Masculino,Preto,1 – 2 salários mínimos,1,0,Nenhuma comorbidade,Não,105.0,168,37.202381,12.80,8.75,12,15
4,17,42,Solteiro,Masculino,Preto,Até 1 salário mínimo,1,0,Nenhuma comorbidade,Não,75.0,165,27.548209,10.00,8.10,10,12


### Média, SD e difference (session 2 - session 1) - TUG

In [4]:
mean1 = df['tug_1'].mean()
stfd1 = df['tug_1'].std()
print(f"Media: {mean1}; SD {stfd1};")

mean2 = df['tug_2'].mean()
stfd2 = df['tug_2'].std()
print(f"Media: {mean2}; SD {stfd2};")

diff_mean = (df['tug_2'] - df['tug_1']).mean()
diff_std = (df['tug_2'] - df['tug_1']).std()
print(f"Diff: {diff_mean}; SD {diff_std};")


Media: 9.53761904761905; SD 2.3878963645055973;
Media: 7.791904761904763; SD 1.8330428774243634;
Diff: -1.7457142857142862; SD 1.5778104177263232;


### Média, SD e difference (session 2 - session 1) - TS

In [5]:
mean1 = df['ts_1'].mean()
stfd1 = df['ts_1'].std()
print(f"Media: {mean1}; SD {stfd1};")

mean2 = df['ts_2'].mean()
stfd2 = df['ts_2'].std()
print(f"Media: {mean2}; SD {stfd2};")

diff_mean = (df['ts_2'] - df['ts_1']).mean()
diff_std = (df['ts_2'] - df['ts_1']).std()
print(f"Diff: {diff_mean}; SD {diff_std};")

Media: 10.428571428571429; SD 3.0260771399855053;
Media: 12.333333333333334; SD 3.2455097185701565;
Diff: 1.9047619047619047; SD 0.9952267030562387;


## **Time Up and Go**

### Organizando os dados do TUG - wide to long

In [6]:
# Selecionar as colunas necessárias
tug_icc = df[['id', 'tug_1', 'tug_2']].copy()

# Transformar do formato wide para long
tug_long = pd.melt(
    tug_icc,
    id_vars='id',
    value_vars=['tug_1', 'tug_2'],
    var_name='day',
    value_name='score'
)

# Organizar a ordem
tug_long['day'] = tug_long['day'].map({
    'tug_1': 1,
    'tug_2': 2
})

tug_long = tug_long.sort_values(['id', 'day']).reset_index(drop=True)

tug_long.head(5)

,id,day,score
0,3,1,8.55
1,3,2,6.08
2,5,1,8.65
3,5,2,6.54
4,8,1,14.20


### Cálculo do ICC (modelo de dois fatores, consistência, medidas médias)

In [7]:
icc_result = pg.intraclass_corr(data=tug_long,
                                targets='id',
                                raters='day',
                                ratings='score')

# print(icc_result) - ICC3 consistência entre medidas
icc_3 = icc_result.set_index('Type').loc[['ICC(C,1)']]
icc_3



,ICC,F,df1,df2,pval,CI95
Type,,,,,,
"ICC(C,1)",0.725286,6.280295,20,20,0.000067,"[0.44, 0.88]"


## **Timed Stands**

### Organizando os dados do TS - wide to long

In [8]:
# Selecionar as colunas necessárias
ts_icc = df[['id', 'ts_1', 'ts_2']].copy()

# Transformar do formato wide para long
ts_long = pd.melt(
    ts_icc,
    id_vars='id',
    value_vars=['ts_1', 'ts_2'],
    var_name='day',
    value_name='score'
)

# Organizar a ordem
ts_long['day'] = ts_long['day'].map({
    'ts_1': 1,
    'ts_2': 2
})

ts_long = ts_long.sort_values(['id', 'day']).reset_index(drop=True)

ts_long.head(5)

,id,day,score
0,3,1,10
1,3,2,13
2,5,1,15
3,5,2,18
4,8,1,6


### Cálculo do ICC (modelo de dois fatores, consistência, medidas médias)

In [9]:
icc_result = pg.intraclass_corr(data=ts_long,
                                targets='id',
                                raters='day',
                                ratings='score')

# print(icc_result) - ICC3 consistência entre medidas
icc_3 = icc_result.set_index('Type').loc[['ICC(C,1)']]
icc_3


,ICC,F,df1,df2,pval,CI95
Type,,,,,,
"ICC(C,1)",0.949698,38.759615,20,20,7.599956e-12,"[0.88, 0.98]"
